# microgpt_torch — microgpt 的 PyTorch 版本

用 PyTorch 从零实现 GPT 的训练和推理，在 32,000 个英文名字上训练一个迷你 Transformer，
然后让它“编造”新的名字。

**与原始版本的区别：**
- `torch.autograd` 替代手工 `Value` 类
- `nn.Module` / `nn.Linear` / `nn.Embedding` 替代手工参数矩阵
- `torch.optim.Adam` 替代手工 Adam
- 整序列并行前向传播（相比原始逐 token 循环更高效）
- 推理时仍用逐 token 自回归生成


In [ ]:
import os
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


## 1. 数据集

32,000 个英文名字，从 GitHub 下载（如果本地没有的话），每行一个名字。


In [ ]:
if not os.path.exists('input.txt'):
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt',
        'input.txt'
    )

docs = [line.strip() for line in open('input.txt') if line.strip()]
random.shuffle(docs)
print(f"num docs: {len(docs)}")


## 2. 分词器

把字母变成数字编号：a→0, b→1, ..., z→25, BOS（开始标记）→26。
词汇表大小 = 26 个字母 + 1 个 BOS = 27。

BOS 的作用：标记序列的开始/结束，让模型知道何时开始/停止生成。


In [ ]:
uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1
print(f"vocab size: {vocab_size}")


## 3. 超参数

控制模型的尺寸：1 层 Transformer、16 维嵌入、4 个注意力头。


In [ ]:
n_layer = 1
n_embd = 16
block_size = 16
n_head = 4
head_dim = n_embd // n_head


## 4. 模型定义

### RMSNorm

均方根归一化：把向量长度缩放到接近 1，防止深层网络中数值爆炸或消失。

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\text{mean}(x^2) + \varepsilon}}$$

其中 $\varepsilon = 10^{-5}$ 防止除以零。与 LayerNorm 的区别：没有减均值（不 center），也没有可学习的缩放参数。


In [ ]:
class RMSNorm(nn.Module):
    def forward(self, x):
        # x: (T, n_embd), dim=-1 对每个 token 的 16 维求均方
        ms = torch.mean(x ** 2, dim=-1, keepdim=True)  # (T, 1)
        return x / torch.sqrt(ms + 1e-5)  # broadcasting: (T, 16) / (T, 1)


### 因果自注意力

**核心公式：**

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

**QKV 三角色：**
- $Q$（Query）：我在找什么样的信息？
- $K$（Key）：我拥有什么样的信息？
- $V$（Value）：如果你选中我，我能给你什么？

**数据流：**
1. 生成 $Q, K, V$：$Q = xW_q$, $K = xW_k$, $V = xW_v$
2. 分头（4 个头，每个 4 维），并行计算
3. 点积得分 $S = QK^T / \sqrt{d_k}$（缩放防止 softmax 过于尖锐）
4. 因果掩码 $M_{ij} = \begin{cases} 0 & i \geq j \\ -\infty & i < j \end{cases}$，$S' = S + M$
5. Softmax：$\text{attn}_{ij} = e^{S'_{ij}} / \sum_k e^{S'_{ik}}$
6. 加权求和：$y_i = \sum_j \text{attn}_{ij} V_j$
7. 合并多头，输出投影 $yW_o$

四个头各自关注不同类型的模式（如开头字母、元音位置、常见结尾等），最后合并。


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        # 四组线性变换，每组内部维护一个 (16, 16) 的权重矩阵
        # 前向等价于 y = x @ W^T，其中 W shape 是 (out_features, in_features)
        # 这里 in=16, out=16，所以 W 是 (16, 16)
        self.wq = nn.Linear(n_embd, n_embd, bias=False)   # Query  生成头
        self.wk = nn.Linear(n_embd, n_embd, bias=False)   # Key    索引头
        self.wv = nn.Linear(n_embd, n_embd, bias=False)   # Value  内容头
        self.wo = nn.Linear(n_embd, n_embd, bias=False)   # 输出投影，混合4个头的结果

    def forward(self, x):
        # x: (T, n_embd), T 是序列长度(≤16), n_embd=16
        T = x.size(0)

        # ========== 1. 生成 Q, K, V ==========
        # 每个 Linear 做 y = x @ W^T
        # x: (T, 16) @ W^T: (16, 16) → q: (T, 16)
        # 注意 W 是 (16, 16)，W^T 是 (16, 16)
        # (T, 16) @ (16, 16) = (T, 16)
        q = self.wq(x)  # Query：当前 token "想找什么"
        k = self.wk(x)  # Key：  当前 token "有什么可被找到的"
        v = self.wv(x)  # Value：当前 token "如果被选中能给什么信息"

        # ========== 2. 分头（split heads）==========
        # 目标：把 16 维分成 4 个头，每个头管 4 维，让不同头关注不同模式
        #
        # 第一步 .view(T, n_head, head_dim) = (T, 4, 4)
        #   把 16 维向量"切断"成 4 段，每段 4 维：
        #   [d0,d1,d2,d3 | d4,d5,d6,d7 | d8,d9,d10,d11 | d12,d13,d14,d15]
        #     └──头0──┘   └──头1──┘   └──头2───┘   └───头3────┘
        #
        # 第二步 .transpose(0, 1) = (4, T, 4)
        #   把 T 维度和 head 维度交换，让头在最前面
        #   这样每个头可以独立且并行地做注意力计算
        #   直观理解：从"第 i 个 token 的 4 个头"变成"第 h 个头在所有 T 个 token 上的值"
        q = q.view(T, n_head, head_dim).transpose(0, 1)  # (4, T, 4)
        k = k.view(T, n_head, head_dim).transpose(0, 1)  # (4, T, 4)
        v = v.view(T, n_head, head_dim).transpose(0, 1)  # (4, T, 4)

        # ========== 3. 计算注意力分数 ==========
        # Q 和 K 做点积：每个 token 的 Q 与所有 token 的 K 求相似度
        #
        # k.transpose(-2, -1): (4, T, 4) → (4, 4, T)
        #   交换最后两维，把 head_dim 放到前面，T 放到最后
        #   这样 q @ k^T 时，最后两维做矩阵乘法：(T, 4) @ (4, T) = (T, T)
        #
        # q @ k^T: (4, T, 4) @ (4, 4, T) = (4, T, T)
        #   结果 att[h, i, j] = 头 h 中 token_i 的 Q 与 token_j 的 K 的点积
        #   含义：token_i 认为 token_j 和自己有多相关
        #
        # * (head_dim ** -0.5): 缩放因子 = 1 / sqrt(4) = 0.5
        #   点积的大小随维度 d_k 线性增长。如果 d_k 大，点积的方差就大，
        #   softmax 后的分布会极度尖锐（接近 one-hot），梯度趋近于 0，
        #   模型就无法有效学习。除以 sqrt(d_k) 让方差恢复到 1。
        att = q @ k.transpose(-2, -1) * (head_dim ** -0.5)
        # att shape: (4, T, T)

        # ========== 4. 因果掩码（Causal Mask）==========
        # 语言模型是自回归的：生成时从左到右逐 token 生成，
        # 所以训练时 token_i 只能看到它自己和之前的 token（j ≤ i），
        # 不能看到未来的 token（j > i）。否则相当于"偷看答案"。
        #
        # torch.triu(..., diagonal=1)：生成上三角矩阵（不含对角线）
        # 用对角线为 1 的上三角矩阵，即主对角线及以下为 0，以上为 -inf
        #
        # 以 T=4 为例：
        #   mask = [[0, -inf, -inf, -inf],
        #           [0,    0, -inf, -inf],
        #           [0,    0,    0, -inf],
        #           [0,    0,    0,    0]]
        #
        # token_0 能看到 [0]         （自己）
        # token_1 能看到 [0, 1]      （自己和前面的）
        # token_2 能看到 [0, 1, 2]
        # token_3 能看到 [0, 1, 2, 3]（所有前面的 + 自己）
        #
        # 加上 -inf 后，softmax(e^{-inf}) ≈ 0，这些位置的注意力权重为 0
        # 广播加法：(4, T, T) + (T, T) → 每个头都加上相同的掩码
        mask = torch.triu(
            torch.full((T, T), float('-inf'), device=x.device),
            diagonal=1
        )
        att = att + mask

        # ========== 5. Softmax 归一化 ==========
        # 对每个 token_i，把它对所有历史 token 的分数变成概率分布
        # dim=-1 对最后一维（T 个 key）做 softmax
        # 结果 att[h, i, :] = token_i 分配给每个历史 token 的注意力权重
        # 权重和 = 1
        att = F.softmax(att, dim=-1)  # (4, T, T)
        # 例如某行 att[h, 2, :] = [0.2, 0.3, 0.5, 0.0, 0.0, ...]
        # token_2 关注自己 20%，关注 token_0 30%，关注 token_1 50%

        # ========== 6. 对 V 加权求和 ==========
        # 用注意力权重对 V 做加权平均：权重越大的 token，其 V 贡献越多
        # att @ v: (4, T, T) @ (4, T, 4) = (4, T, 4)
        # y[h, i, :] = sum_j att[h, i, j] * v[h, j, :]
        # 即 token_i 从所有历史 token_j 中提取信息，权重由注意力分数决定
        y = att @ v  # (4, T, 4)

        # ========== 7. 合并多头 ==========
        # .transpose(0, 1): (4, T, 4) → (T, 4, 4)
        #   把头和序列维度换回来，恢复成"第 i 个 token 的 4 个头"
        # .contiguous(): transpose 只改了 stride 元数据，内存不连续，
        #   view 要求内存连续，所以先 .contiguous() 重新排布内存
        # .view(T, n_embd): (T, 4, 4) → (T, 16)
        #   把 4 个头 × 4 维拼回 16 维向量
        #   是分头的逆操作：
        #   [头0(4d), 头1(4d), 头2(4d), 头3(4d)] → [16维向量]
        y = y.transpose(0, 1).contiguous().view(T, n_embd)  # (T, 16)

        # ========== 8. 输出投影 ==========
        # wo: Linear(16, 16)，再做一次线性变换混合 4 个头的信息
        # 每个头可能关注了不同特征（元音位置、辅音模式、常见结尾等），
        # 这一步把它们融合成一个统一的表示
        return self.wo(y)  # (T, 16) → (T, 16)


### MLP（多层感知机）

**公式：**

$$\text{MLP}(x) = W_2 \cdot \text{ReLU}(W_1 \cdot x)$$

$$\text{ReLU}(x) = \max(0, x)$$

数据流：$x \in \mathbb{R}^{16} \xrightarrow{W_1} \mathbb{R}^{64} \xrightarrow{\text{ReLU}} \mathbb{R}^{64} \xrightarrow{W_2} \mathbb{R}^{16}$

扩展到 4 倍宽度给模型更大的“思考空间”，ReLU 引入非线性（否则多层线性=单层），再压缩回 16 维。


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # fc1: (16, 64), fc2: (64, 16)，无偏置
        self.fc1 = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.fc2 = nn.Linear(4 * n_embd, n_embd, bias=False)

    def forward(self, x):
        # x: (T, 16) @ W1^T: (16, 64) → (T, 64)
        x = self.fc1(x)
        x = F.relu(x)  # 负值归零，引入非线性
        # (T, 64) @ W2^T: (64, 16) → (T, 16)
        x = self.fc2(x)
        return x


### Transformer 块

**残差连接 + 预归一化架构：**

$$x' = x + \text{Attention}(\text{Norm}(x))$$

$$x'' = x' + \text{MLP}(\text{Norm}(x'))$$

数据流：$x \xrightarrow{\text{RMSNorm}} \xrightarrow{\text{Attn}} \xrightarrow{+} \xrightarrow{\text{RMSNorm}} \xrightarrow{\text{MLP}} \xrightarrow{+}$

残差连接的两个好处：
- **梯度直通**：反向传播时梯度可以跳过子层，防止梯度消失
- **增量学习**：每层只需学“在原始信息上做什么修改”，而非从零开始


In [ ]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn_norm = RMSNorm()      # 注意力前归一化
        self.attn = CausalSelfAttention()  # 因果自注意力
        self.mlp_norm = RMSNorm()       # MLP 前归一化
        self.mlp = MLP()                # 前馈网络

    def forward(self, x):
        # 注意力子层：先归一化，再做注意力，最后残差
        x = x + self.attn(self.attn_norm(x))
        # MLP 子层：先归一化，再过 MLP，最后残差
        x = x + self.mlp(self.mlp_norm(x))
        return x


### GPT 模型

**完整数据流：**

$$\text{token}_i \xrightarrow{W_e} h_i \xrightarrow{+ W_p[i]} x_i \xrightarrow{\text{Norm}} \xrightarrow{\text{Block}_1} \cdots \xrightarrow{\text{Block}_{n_L}} \xrightarrow{W_{\text{head}}} \text{logits}_i$$

$$x_i = W_e[t_i] + W_p[i] \quad \text{(Token 嵌入 + 位置嵌入)}$$

$$\text{logits}_i = W_{\text{head}}(\text{Block}(\cdots \text{Block}(\text{Norm}(x_i)) \cdots ))$$

其中 $W_e \in \mathbb{R}^{27 \times 16}$ 是 token 嵌入表，$W_p \in \mathbb{R}^{16 \times 16}$ 是位置嵌入表，
$W_{\text{head}} \in \mathbb{R}^{16 \times 27}$ 是输出投影。


In [ ]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd)   # (27, 16)
        self.wpe = nn.Embedding(block_size, n_embd)  # (16, 16)
        self.embed_norm = RMSNorm()
        self.layers = nn.ModuleList([Block() for _ in range(n_layer)])
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)  # (16, 27)

    def forward(self, idx):
        T = idx.size(0)
        tok_emb = self.wte(idx)                               # (T, 16)
        pos_emb = self.wpe(torch.arange(T, device=idx.device))  # (T, 16)
        x = self.embed_norm(tok_emb + pos_emb)                 # (T, 16)
        for layer in self.layers:
            x = layer(x)
        return self.lm_head(x)  # (T, 27)


### 实例化模型

**参数统计：**
- wte: $27 \times 16 = 432$
- wpe: $16 \times 16 = 256$
- attention: $4 \times 16 \times 16 = 1024$
- MLP: $64 \times 16 + 16 \times 64 = 2048$
- lm\_head: $27 \times 16 = 432$
- **合计: 4192 个参数**

GPT-3 有 1750 亿参数，但算法完全一样，区别只在规模。


In [ ]:
model = GPT()
print(f"num params: {sum(p.numel() for p in model.parameters())}")


## 5. 训练

**损失函数（交叉熵）：**

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^N \log p(y_i \mid x_i)$$

其中 $p(y_i \mid x_i) = \text{softmax}(\text{logits}_i)_{y_i} = \dfrac{e^{\text{logits}_i[y_i]}}{\sum_{j=1}^{27} e^{\text{logits}_i[j]}}$

直观理解：模型预测正确的概率越高，损失越低。完美预测时损失为 0，随机猜测时 $-\log(1/27) \approx 3.3$。

**学习率调度（线性衰减）：**

$$\alpha_t = \alpha_0 \cdot \left(1 - \frac{t}{T}\right), \quad \alpha_0 = 0.01, T = 1000$$

训练后期减小步幅，让模型精细调整而不是大幅跳动。

**Adam 优化器：**

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t \quad \text{（一阶矩，梯度方向平滑）}$$

$$v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2 \quad \text{（二阶矩，自适应步长）}$$

$$\theta_{t+1} = \theta_t - \alpha_t \cdot \frac{m_t}{\sqrt{v_t} + \varepsilon}$$

每一步：选一个名字 → 前向传播算损失 → 反向传播算梯度 → Adam 更新参数。


In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    betas=(0.85, 0.99),
    eps=1e-8
)
num_steps = 1000

for step in range(num_steps):
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    tokens = tokens[:block_size + 1]

    x = torch.tensor(tokens[:-1])
    y = torch.tensor(tokens[1:])

    logits = model(x)
    loss = F.cross_entropy(logits, y)

    optimizer.zero_grad()
    loss.backward()

    lr_t = 0.01 * (1 - step / num_steps)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr_t

    optimizer.step()

    print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.item():.4f}", end='\r')

print()


## 6. 推理生成

**自回归生成：** 从 BOS 出发，逐 token 预测、采样，把新 token 拼回序列继续，遇到 BOS 停止。

**Softmax + 温度采样：**

$$p_j = \frac{e^{z_j / T}}{\sum_{k=1}^{27} e^{z_k / T}}$$

其中 $T$ 是温度，$z_j$ 是第 $j$ 个 token 的 logit：
- $T \to 0$：分布趋于 one-hot（每次都选最高概率，确定但缺乏多样性）
- $T = 1$：原始概率分布
- $T > 1$：分布更平滑（更随机，可能产生意想不到的有趣结果）


In [ ]:
temperature = 0.5
print("--- inference (new, hallucinated names) ---")

@torch.no_grad()
def generate(temperature=0.5):
    model.eval()
    tokens = [BOS]
    for _ in range(block_size):
        idx = torch.tensor(tokens[-block_size:])
        logits = model(idx)[-1] / temperature
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1).item()
        if next_token == BOS:
            break
        tokens.append(next_token)
    return ''.join(uchars[t] for t in tokens[1:])

for i in range(20):
    print(f"sample {i+1:2d}: {generate(temperature)}")
